# UEK Schedule Scraper
### Anna Chabrajska (237095, Data Science w naukach społecznych, EIDSS1-2411, 1 stopień 2 rok)

### Import bibliotek

In [21]:
import getpass
from bs4 import BeautifulSoup
import requests
import os
import json
import pandas as pd

## Uwierzytelnianie i pobieranie danych

### Uwierzytelnianie użytkownika

In [22]:
username = input("Podaj login: ")
password = getpass.getpass("Podaj hasło: ")

### Wysłanie żądania dostępu do zasobu (strony WWW)

In [23]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [24]:
plan_id = 259981
okres = 2
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={plan_id}&okres={okres}"

response = requests.get(url, headers=headers, auth=(username, password))
response.encoding = 'utf-8'

if response.status_code == 200:
    print("Sukces.")
    
    page_dom = BeautifulSoup(response.text, 'html.parser')
    tables = page_dom.select("table")
    
    if tables:
        print("\nFragment Twojego planu:")
        print(tables[0].get_text(separator=" | ", strip=True)[:250] + "...")

elif response.status_code == 401:
    print("Błąd.")
    
else:
    print(f"Inny błąd. Kod statusu: {response.status_code}")

Sukces.

Fragment Twojego planu:
Termin | Dzień, godzina | Przedmiot | Typ | Nauczyciel | Sala | 2026-02-23 | Pn 09:45 - 11:15 (2g.) | Język obcy | lektorat | Wybierz swoją grupę językową | 2026-02-23 | Pn 11:30 - 13:00 (2g.) | Podstawy zarządzania | wykład | dr Małgorzata Marchewka...


## Parsowanie kodu HTML

In [25]:
page_dom = BeautifulSoup(response.text, 'html.parser')
    
grupa_info = page_dom.select_one("div.grupa")
nazwa_grupy = grupa_info.get_text(strip=True) if grupa_info else "Brak danych"
print(f"Plan dla grupy: {nazwa_grupy}")

tables = page_dom.select("table")
print(f"Znaleziono {len(tables)} tabel na stronie.")

if tables:
    plan_table = tables[0]
    
    rows = plan_table.select("tr")
    print(f"Liczba wszystkich wierszy (razem z nagłówkiem): {len(rows)}")
    
    header_row = rows.pop(0) 
    
    print(f"Liczba wierszy z samymi zajęciami do pobrania: {len(rows)}")

Plan dla grupy: EIDSS1-2411
Znaleziono 1 tabel na stronie.
Liczba wszystkich wierszy (razem z nagłówkiem): 240
Liczba wierszy z samymi zajęciami do pobrania: 239


### Analiza struktury wiersza
| Składowa | Nazwa zmiennej | Indeks w kodzie |
| :--- | :--- | :--- |
| Data zjazdu | termin | cells[0] |
| Dzień tygodnia i godziny | dzien_godzina | cells[1] |
| Nazwa zajęć | przedmiot | cells[2] |
| Forma zajęć (np. wykład) | typ | cells[3] |
| Prowadzący | nauczyciel | cells[4] |
| Miejsce (sala/budynek) | sala | cells[5] |

In [26]:
all_classes = []

for row in rows:
    cells = row.select("td")
    
    if len(cells) >= 6:
        class_data = {}
        
        class_data["termin"] = cells[0].get_text(strip=True)
        class_data["dzien_godzina"] = cells[1].get_text(strip=True)
        class_data["przedmiot"] = cells[2].get_text(strip=True)
        class_data["typ"] = cells[3].get_text(strip=True)
        class_data["nauczyciel"] = cells[4].get_text(strip=True)
        class_data["sala"] = cells[5].get_text(strip=True)
        
        all_classes.append(class_data)

print(f"Pomyślnie przetworzono {len(all_classes)} zajęć.")

if all_classes:
    print("\nPierwsze zajęcia na liście:")
    print(all_classes[0])

Pomyślnie przetworzono 222 zajęć.

Pierwsze zajęcia na liście:
{'termin': '2026-02-23', 'dzien_godzina': 'Pn 09:45 - 11:15 (2g.)', 'przedmiot': 'Język obcy', 'typ': 'lektorat', 'nauczyciel': '', 'sala': 'Wybierz swoją grupę językową'}


In [27]:
df = pd.DataFrame(all_classes)
df.insert(0, 'grupa', nazwa_grupy)

display(df)

,grupa,termin,dzien_godzina,przedmiot,typ,nauczyciel,sala
0,EIDSS1-2411,2026-02-23,Pn 09:45 - 11:15 (2g.),Język obcy,lektorat,,Wybierz swoją grupę językową
1,EIDSS1-2411,2026-02-23,Pn 11:30 - 13:00 (2g.),Podstawy zarządzania,wykład,dr Małgorzata Marchewka,Paw.F 107
2,EIDSS1-2411,2026-02-23,Pn 13:15 - 14:45 (2g.),Sieci neuronowe i uczenie maszynowe,wykład,dr Sebastian Baran,Paw.F 516
3,EIDSS1-2411,2026-02-23,Pn 17:30 - 20:00 (3g.),Nadzorowane uczenie statystyczne,laboratorium,mgr Łukasz Cholewa,"Paw.A 117 lab. Win10, Office21"
4,EIDSS1-2411,2026-02-24,Wt 09:45 - 11:15 (2g.),Język obcy,lektorat,,Wybierz swoją grupę językową
...,...,...,...,...,...,...,...
217,EIDSS1-2411,2026-06-16,Wt 08:00 - 17:30 (11g.),,rezerwacja,Językowe Centrum,.
218,EIDSS1-2411,2026-06-18,Cz 11:30 - 13:00 (2g.),,egzamin,prof. UEK dr hab. Justyna Wróblewska,"Paw.A 09 lab. Win8.1, Office21"
219,EIDSS1-2411,2026-06-19,Pt 09:45 - 11:15 (2g.),,egzamin,dr Sebastian Baran,Paw.C sala A
220,EIDSS1-2411,2026-06-19,Pt 11:30 - 13:00 (2g.),,egzamin,dr Sebastian Baran,"Paw.A 116 lab.Win8.1, Office21, Paw.A 117 lab...."


## Zapis i eksport danych

In [28]:
if not os.path.exists("./plany_zajec"):
    os.mkdir("./plany_zajec")

In [29]:
with open("./plany_zajec/moj_plan.json", "w", encoding="UTF-8") as jf:
    json.dump(all_classes, jf, indent=4, ensure_ascii=False)

In [30]:
with open("./plany_zajec/moj_plan.json", "w", encoding="UTF-8") as jf:
    json.dump(all_classes, jf, indent=4, ensure_ascii=False)